# Iris Classification with RandomForestClassifier
Author: Manuel Eugenio Morocho Cayamcela, PhD

A random forest combines predictions from multiple decision trees. This standalone notebook contains the random forest section from the k-NN lesson: train a classifier, evaluate it, inspect feature importances, and visualize one tree.

If needed, install dependencies once with `%pip install pandas matplotlib seaborn scikit-learn`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Load the Iris dataset
Read the Iris CSV directly from GitHub using pandas. An internet connection is required.

In [ ]:
github_url = "https://raw.githubusercontent.com/eugeniomorocho/ArtificialIntelligence/main/Datasets/iris.csv"
iris = pd.read_csv(github_url)
iris.head()

## Prepare the data
Remove the identifier and reserve a stratified 20% test set. Train only on the training subset; fitting on the complete dataset would leak test examples into the model and invalidate evaluation.

In [ ]:
iris = iris.drop(columns="Id")

In [ ]:
features = iris.drop(columns="Species")
target = iris["Species"]
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=42
)

## Train a random forest
`n_estimators=10` creates ten decision trees, preserving the small forest used in the original lesson. A fixed seed makes training reproducible.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(n_estimators=10, random_state=42)
classifier.fit(features_train, target_train)

## Evaluate the classifier
Compute accuracy and per-class precision, recall, and F1-score on held-out examples. In the confusion matrix, rows are actual classes and columns are predictions.

In [ ]:
predictions = classifier.predict(features_test)
print(f"Test accuracy: {accuracy_score(target_test, predictions):.3f}")
print(classification_report(target_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    target_test, predictions, labels=classifier.classes_, cmap="Blues"
)
plt.title("Iris classification: confusion matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Feature importances
The forest reports each feature's normalized contribution to reducing impurity during training. Higher values indicate greater use in this fitted model; they do not establish causation, and correlated features can share importance.

In [ ]:
feature_importances = pd.DataFrame({
    "feature": features.columns,
    "importance": classifier.feature_importances_,
}).sort_values("importance", ascending=False)
display(feature_importances)
sns.barplot(data=feature_importances, x="importance", y="feature", color="steelblue")
plt.title("Feature importances for Iris classification")
plt.tight_layout()
plt.show()
print("The two most important features are:")
display(feature_importances.head(2))

## Visualize one decision tree
Display the third tree in the forest. Each node shows a split or leaf, impurity, sample count, and class information. This is one component of the ensemble; the full classifier combines all ten trees.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(
    classifier.estimators_[2],
    feature_names=features.columns.tolist(),
    class_names=classifier.classes_.tolist(),
    filled=True,
    rounded=True,
)
plt.title("Third decision tree in the random forest")
plt.show()